In [ ]:
%pip install yfinance pandas mysql-connector-python

# Prepare Tweet Data for MySQL

Clean historical tweets for 2020–2022 by removing duplicate tweet IDs,
handling missing records, and removing embedded line breaks

Save a UTF-8 CSV for loading into MySQL
Ticker matching and buy/sell word counts are performed later in SQL

### Part 1  Load local CSV data into AWS RDS MySQL using Python and query it from Python:

### Download Data

In [ ]:
from pathlib import Path
import yfinance as yf

# CSV files will be saved in a data folder.
data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

tickers = ["AAPL", "AMZN", "GOOGL", "MSFT", "TSLA"]

for ticker in tickers:
    stock = yf.download(
        ticker,
        start="2020-01-01",
        end="2023-01-01",
        interval="1d",
        auto_adjust=False,
        multi_level_index=False,
        progress=False
    )

    if stock.empty:
        raise ValueError(f"No data downloaded for {ticker}")

    # Move the date from the index into a regular column.
    stock = stock.reset_index()

    # Keep the columns we need for the assignment.
    stock = stock[
        ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    ]

    stock["Date"] = stock["Date"].dt.strftime("%Y-%m-%d")

    file_path = data_folder / f"{ticker}.csv"
    stock.to_csv(file_path, index=False)

    print(
        ticker,
        "| Rows:", len(stock),
        "| First date:", stock["Date"].min(),
        "| Last date:", stock["Date"].max()
    )

In [ ]:
import pandas as pd

apple = pd.read_csv("data/AAPL.csv")

print(apple.head())
print(apple.tail())
print(apple.isna().sum())


###Missing dates are weekends and market holidays

### Connect Python to AWS RDS Server

In [ ]:
import mysql.connector
import pandas as pd

connection = mysql.connector.connect(
    host="mysql-db.cen80qo2cqj1.us-east-1.rds.amazonaws.com",
    user="admin",
    password="gbs%20#MSBA5",
    port=3306,
    connection_timeout=20
)

cursor = connection.cursor()

print("Connected to AWS RDS!")

### Create database 


In [ ]:
cursor.execute("CREATE DATABASE IF NOT EXISTS assignment3")
cursor.execute("USE assignment3")

print("Using assignment3 database")

### Create on table for each comapny

In [ ]:
tickers = ["AAPL", "AMZN", "GOOGL", "MSFT", "TSLA"]

for ticker in tickers:
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS `{ticker}` (
            stock_date DATE PRIMARY KEY,
            open_price DOUBLE,
            high_price DOUBLE,
            low_price DOUBLE,
            close_price DOUBLE,
            adjusted_close DOUBLE,
            volume BIGINT
        )
    """)

print("Five stock tables created")

### Read your local CSV files and insert the data

In [ ]:
for ticker in tickers:
    stock = pd.read_csv(f"data/{ticker}.csv")

    # Check the data before inserting it.
    if stock.isna().any().any():
        raise ValueError(f"{ticker} contains missing values")

    if stock["Date"].duplicated().any():
        raise ValueError(f"{ticker} contains duplicate dates")

    # Convert each row into values MySQL can accept.
    rows = []

    for _, row in stock.iterrows():
        rows.append((
            row["Date"],
            float(row["Open"]),
            float(row["High"]),
            float(row["Low"]),
            float(row["Close"]),
            float(row["Adj Close"]),
            int(row["Volume"])
        ))

    insert_sql = f"""
        INSERT INTO `{ticker}` (
            stock_date,
            open_price,
            high_price,
            low_price,
            close_price,
            adjusted_close,
            volume
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            open_price = VALUES(open_price),
            high_price = VALUES(high_price),
            low_price = VALUES(low_price),
            close_price = VALUES(close_price),
            adjusted_close = VALUES(adjusted_close),
            volume = VALUES(volume)
    """

    cursor.executemany(insert_sql, rows)
    connection.commit()

    print(f"{ticker}: loaded {len(rows)} records")

### Verify records inside RDS

In [ ]:
for ticker in tickers:
    cursor.execute(f"""
        SELECT COUNT(*), MIN(stock_date), MAX(stock_date)
        FROM `{ticker}`
    """)

    count, first_date, last_date = cursor.fetchone()

    print(
        ticker,
        "| Records:", count,
        "| First date:", first_date,
        "| Last date:", last_date
    )

### Month with the highest total trading volume for each company

In [ ]:
highest_volume_results = []

for ticker in tickers:
    cursor.execute(f"""
        SELECT
            YEAR(stock_date) AS year,
            MONTH(stock_date) AS month,
            SUM(volume) AS monthly_volume
        FROM `{ticker}`
        WHERE stock_date >= '2020-01-01'
          AND stock_date < '2023-01-01'
        GROUP BY YEAR(stock_date), MONTH(stock_date)
        ORDER BY monthly_volume DESC
    """)

    monthly_results = cursor.fetchall()
    highest_volume = monthly_results[0][2]

    for year, month, monthly_volume in monthly_results:
        if monthly_volume == highest_volume:
            highest_volume_results.append({
                "Ticker": ticker,
                "Year": year,
                "Month": month,
                "Monthly_Volume": int(monthly_volume)
            })

results = pd.DataFrame(highest_volume_results)

print(results.to_string(index=False))

In [ ]:
cursor.close()
connection.close()

print("Connection closed")

### Part 2 Load local CSV data to the local MySQL server and query it in MySQL Workbench:

In [ ]:
import pandas as pd

tweets_sample = pd.read_csv(
    "data/tweets_raw.csv",
    nrows=5
)

print("Columns:")
print(tweets_sample.columns.tolist())

print("\nSample records:")
print(tweets_sample.to_string(index=False))

print("\nData types:")
print(tweets_sample.dtypes)

### Cleaning: read in chunks so it doesn't overload

In [ ]:
# Preserve emojis and punctuation. 
# Remove embedded line breaks to make importing easier and deduplicate by tweet ID, 
# so the same tweet is not counted repeatedly simply because it appeared under different source tickers.
# Have not filtered using the Ticker column. MySQL will identify your five tickers from Text using REGEXP.



from pathlib import Path
import pandas as pd
import csv

# Read and save files inside the project's data folder
data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

input_file = data_folder / "tweets_raw.csv"
output_file = data_folder / "tweets_clean.csv"

seen_ids = set()
total_saved = 0
first_chunk = True

for chunk in pd.read_csv(
    input_file,
    usecols=["ID", "Date", "Text"],
    dtype={"ID": "string", "Text": "string"},
    chunksize=100000
):
    # Convert timestamps to numbers.
    chunk["Date"] = pd.to_numeric(chunk["Date"], errors="coerce")

    # Remove records missing essential information.
    chunk = chunk.dropna(subset=["ID", "Date", "Text"])

    # Keep January 1, 2020 through December 31, 2022 (UTC).
    chunk = chunk[
        (chunk["Date"] >= 1577836800) &
        (chunk["Date"] < 1672531200)
    ].copy()

    chunk["Date"] = chunk["Date"].astype("int64")

    # A tweet may appear under several source tickers.
    # Keep each tweet ID only once.
    chunk = chunk.drop_duplicates(subset=["ID"])
    chunk = chunk[~chunk["ID"].isin(seen_ids)].copy()
    seen_ids.update(chunk["ID"])

    # Replace embedded line breaks, tabs, and null characters.
    chunk["Text"] = (
        chunk["Text"]
        .str.replace(r"[\r\n\t]+", " ", regex=True)
        .str.replace("\x00", "", regex=False)
        .str.strip()
    )

    chunk = chunk[chunk["Text"] != ""]

    # Save a consistently quoted UTF-8 CSV.
    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding="utf-8",
        quoting=csv.QUOTE_ALL,
        doublequote=True,
        lineterminator="\n"
    )

    total_saved += len(chunk)
    first_chunk = False

print("Saved:", output_file)
print("Unique tweets saved:", total_saved)

In [ ]:
clean_sample = pd.read_csv(
    "data/tweets_clean.csv",
    dtype={"ID": "string"},
    nrows=5
)

print(clean_sample.to_string(index=False))

### Make path for SQL to load

In [ ]:
# Find full path to cleaned CSV

from pathlib import Path

print(Path("data/tweets_clean.csv").resolve().as_posix())

In [ ]:
from pathlib import Path
import pandas as pd

file_path = Path(
    "C:/Users/chaug/Desktop/Big Data Class/Assignment 3/data/tweets_clean.csv"
)

print("File exists:", file_path.is_file())
print("File size in bytes:", file_path.stat().st_size)

with file_path.open(encoding="utf-8") as file:
    for _ in range(3):
        print(repr(file.readline()[:250]))

clean_check = pd.read_csv(file_path, dtype={"ID": "string"})

print("Rows:", len(clean_check))
print("Columns:", clean_check.columns.tolist())

In [ ]:
from pathlib import Path

file_path = Path("data/trends/AAPL_2020_Q1.csv")

with file_path.open(encoding="utf-8-sig") as file:
    for _ in range(8):
        print(file.readline().rstrip())